# 📺 Day 05a: HLD — Netflix / YouTube (Video Streaming)

---

## 🎯 What You'll Master Today
- Video upload & transcoding pipeline
- Adaptive bitrate streaming (HLS/DASH)
- CDN for video delivery
- Recommendation engine placement

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  VIDEO STREAMING — ARCHITECTURE                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Two Systems:                                                    ║
║  1. VIDEO UPLOAD (write path) — async, heavy processing         ║
║  2. VIDEO WATCHING (read path) — real-time, low latency         ║
║                                                                   ║
║  Upload Pipeline:                                                ║
║  Creator ──► Upload ──► Object Store ──► Transcoding ──► CDN   ║
║                              (raw)         (async)      (serve) ║
║                                                                   ║
║  Watch Flow:                                                     ║
║  User ──► API (get video metadata + CDN URL) ──► CDN ──► User  ║
║                                                                   ║
║  Transcoding (the magic):                                        ║
║  Raw video (4K, 10GB) → multiple versions:                      ║
║  ┌─────────────────────────────────────────────┐                ║
║  │ Resolution │ Bitrate │ Codec  │ Size (1hr)  │                ║
║  │ 2160p (4K) │ 15 Mbps│ H.264  │ ~6.75 GB    │                ║
║  │ 1080p      │ 5 Mbps │ H.264  │ ~2.25 GB    │                ║
║  │ 720p       │ 2.5 Mbps│ H.264 │ ~1.1 GB     │                ║
║  │ 480p       │ 1 Mbps │ H.264  │ ~450 MB     │                ║
║  │ 360p       │ 0.5 Mbps│ H.264 │ ~225 MB     │                ║
║  └─────────────────────────────────────────────┘                ║
║  + Different codecs (H.264, H.265, VP9, AV1)                   ║
║  + Different containers (MP4, WebM)                              ║
║  = ~20-30 versions per video!                                    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Scale Estimation
# ============================================================

print("""
📊 YOUTUBE / NETFLIX SCALE

YouTube:
  • 500 hours of video uploaded EVERY MINUTE
  • 1 billion hours of video watched per DAY
  • 2 billion monthly active users

Netflix:
  • 230M subscribers
  • 17,000+ titles
  • ~100 PB of video content
  • Peak: 15% of global internet bandwidth

Upload estimation (YouTube-like):
  500 hrs/min × 60 = 30,000 hrs/hr = 720,000 hrs/day
  Average video 500MB (mixed quality originals)
  720,000 hrs × 500MB/hr ≈ 360 TB raw upload/day
  After transcoding (20 versions): ~7 PB/day storage growth!

Watch estimation:
  1B hours/day at average 2.5 Mbps = massive CDN bandwidth
  1B hrs × 3600 sec × 2.5 Mbps / 8 ≈ 1.125 EB/day egress!

💡 This is why YouTube/Netflix run their OWN CDN (Open Connect)
""")

In [ ]:
# ============================================================
# 2. Video Upload & Transcoding Pipeline
# ============================================================

print("""
📤 VIDEO UPLOAD PIPELINE

Step 1: Upload
  • Resumable upload (support for large files, network drops)
  • Client → pre-signed URL → S3/GCS direct upload
  • Chunked upload: split into 5MB chunks, upload in parallel

Step 2: Transcoding (HEAVIEST part)
  • Message queue (Kafka) triggers transcoding workers
  • DAG (Directed Acyclic Graph) of tasks:
    
    Raw Video → [Split into segments] → [Parallel transcode]
                                              │
                    ┌─────────────┬────────────┼────────────┐
                    │             │            │            │
                 4K encode    1080p encode  720p encode  480p encode
                    │             │            │            │
                    └─────────────┴────────────┼────────────┘
                                               │
                                    [Generate manifests]
                                    (HLS .m3u8 / DASH .mpd)
                                               │
                                    [Upload to CDN origin]

  • Each segment: 2-10 seconds of video
  • Allows parallel processing AND adaptive streaming

Step 3: Post-processing
  • Generate thumbnails (multiple timestamps)
  • Extract audio track
  • Auto-generate subtitles (speech-to-text)
  • Content moderation (nudity, violence detection)
  • Update DB: video status = 'ready'
""")

In [ ]:
# ============================================================
# 3. Adaptive Bitrate Streaming (ABR)
# ============================================================

print("""
📡 ADAPTIVE BITRATE STREAMING (ABR)

Problem: Users have varying bandwidth. Playing 4K on 3G = buffering.
Solution: Switch quality dynamically based on bandwidth.

How it works:
  1. Video split into small segments (2-10 seconds each)
  2. Each segment encoded at EVERY quality level
  3. Client measures current bandwidth
  4. Client requests next segment at appropriate quality
  
  Timeline:  [seg1-1080p] [seg2-720p] [seg3-720p] [seg4-1080p]
  Bandwidth:  ████████    ████        ████        ████████████
  (seamless quality changes — user doesn't notice!)

Two Standards:
┌──────────────────────────────────────────────────┐
│ Feature     │ HLS (Apple)     │ DASH (MPEG)      │
├──────────────────────────────────────────────────┤
│ Manifest    │ .m3u8           │ .mpd (XML)       │
│ Segments    │ .ts or .fmp4    │ .mp4 / .webm     │
│ Codec       │ H.264, H.265   │ Any              │
│ DRM         │ FairPlay        │ Widevine, PlayReady│
│ Used by     │ Apple, most     │ YouTube, Netflix │
│ Browser     │ Safari native   │ Via MSE API      │
└──────────────────────────────────────────────────┘

HLS Manifest (.m3u8) example:
  #EXTM3U
  #EXT-X-STREAM-INF:BANDWIDTH=5000000,RESOLUTION=1920x1080
  1080p/playlist.m3u8
  #EXT-X-STREAM-INF:BANDWIDTH=2500000,RESOLUTION=1280x720
  720p/playlist.m3u8
  #EXT-X-STREAM-INF:BANDWIDTH=1000000,RESOLUTION=854x480
  480p/playlist.m3u8

💡 Netflix uses DASH + their own Open Connect CDN
   YouTube uses DASH + Google's global CDN
""")

In [ ]:
# ============================================================
# 4. Full Architecture
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  VIDEO STREAMING FULL ARCHITECTURE                           ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  UPLOAD PATH:                                                  ║
║  Creator → Upload API → S3 (raw) → Kafka → Transcoding       ║
║                                       Workers → S3 (encoded)  ║
║                                                → CDN origin   ║
║                                                                ║
║  WATCH PATH:                                                   ║
║  User → API (metadata + CDN URL) → CDN edge → segments       ║
║            │                                                   ║
║            └→ Recommendation Service → personalized home page ║
║                                                                ║
║  Component Detail:                                             ║
║  ┌──────────────┐  ┌────────────┐  ┌──────────────┐          ║
║  │ API Servers  │  │ Redis      │  │ Elastic      │          ║
║  │ (metadata,   │  │ (session,  │  │ Search       │          ║
║  │  search,     │  │  view      │  │ (title,      │          ║
║  │  recomm)     │  │  history)  │  │  tags)       │          ║
║  └──────────────┘  └────────────┘  └──────────────┘          ║
║                                                                ║
║  ┌──────────────┐  ┌────────────┐  ┌──────────────┐          ║
║  │ SQL DB       │  │ Cassandra  │  │ S3           │          ║
║  │ (users,      │  │ (view      │  │ (video       │          ║
║  │  videos      │  │  history,  │  │  segments,   │          ║
║  │  metadata)   │  │  analytics)│  │  thumbnails) │          ║
║  └──────────────┘  └────────────┘  └──────────────┘          ║
║                                                                ║
║  ┌──────────────────────────────────────────────┐             ║
║  │                    CDN                       │             ║
║  │  • Netflix: Open Connect (ISP-hosted caches) │             ║
║  │  • YouTube: Google's global edge network     │             ║
║  │  • Key: pre-position popular content         │             ║
║  │  • 95%+ of traffic served from CDN edge      │             ║
║  └──────────────────────────────────────────────┘             ║
║                                                                ║
║  Recommendation Engine:                                       ║
║  • Collaborative filtering (users who watched X also...)     ║
║  • Content-based (genre, actors, director similarity)        ║
║  • ML models trained offline → serve via API                 ║
║  • A/B testing for ranking algorithms                        ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design YouTube/Netflix | Upload→transcode→CDN. ABR streaming. Segment-based delivery |
| 2 | What is transcoding? | Convert raw video to multiple resolutions/codecs/containers. 20+ versions |
| 3 | How does adaptive bitrate work? | Video split into segments. Client picks quality per segment based on bandwidth |
| 4 | HLS vs DASH? | HLS=Apple(.m3u8), DASH=standard(.mpd). Both segment-based ABR |
| 5 | Why CDN is critical? | 95% traffic from edge. Netflix pre-positions at ISP level |
| 6 | How to handle upload of large video? | Resumable chunked upload. Pre-signed URLs. Parallel chunk upload |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Upload is async — transcoding takes minutes to hours  │
## │  • Split video into 2-10s segments for ABR               │
## │  • Each segment at every quality = seamless switching    │
## │  • CDN serves 95%+ of traffic (open connect at ISPs)    │
## │  • HLS (.m3u8) + DASH (.mpd) are the two standards      │
## │  • Storage is massive → tiered (hot/warm/cold)           │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **HLD — Uber / Ride Sharing**